# identity-risk-engine Demo
End-to-end workflow: synthetic data generation, composite model training, curves, cohort analysis, threshold tuning, and SQL examples.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import types

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("/scratch/fkalghan/circuit_discovery_and_supression/graphs_ai_psych/identity-risk-engine")

SRC_ROOT = PROJECT_ROOT / "src"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

try:
    from identity_risk_engine.synthetic_data_generator import generate_synthetic_login_data, train_test_time_split
    from identity_risk_engine.composite_scorer import CompositeRiskScorer
except Exception as pkg_exc:
    pkg_name = "identity_risk_engine"
    pkg_dir = SRC_ROOT / pkg_name

    if pkg_name not in sys.modules:
        pkg = types.ModuleType(pkg_name)
        pkg.__path__ = [str(pkg_dir)]
        sys.modules[pkg_name] = pkg

    def _load_submodule(name: str):
        full = f"{pkg_name}.{name}"
        if full in sys.modules:
            return sys.modules[full]
        spec = importlib.util.spec_from_file_location(full, pkg_dir / f"{name}.py")
        module = importlib.util.module_from_spec(spec)
        sys.modules[full] = module
        assert spec and spec.loader
        spec.loader.exec_module(module)
        return module

    synthetic_mod = _load_submodule("synthetic_data_generator")
    composite_mod = _load_submodule("composite_scorer")

    generate_synthetic_login_data = synthetic_mod.generate_synthetic_login_data
    train_test_time_split = synthetic_mod.train_test_time_split
    CompositeRiskScorer = composite_mod.CompositeRiskScorer

from dashboard.metrics_dashboard import build_metrics_dashboard, threshold_metrics_table


In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_curve, roc_auc_score, auc

df = generate_synthetic_login_data(num_users=420, num_sessions=18000, attack_ratio=0.22, seed=13)
train_df, test_df = train_test_time_split(df, test_ratio=0.25)
len(df), len(train_df), len(test_df), df['attack_type'].value_counts().to_dict()


In [ ]:
model = CompositeRiskScorer(random_state=13)
model.fit(train_df, target_col='is_attack')
test_scored = test_df.copy()
test_scored['risk_score'] = model.predict_proba(test_scored)[:, 1]
test_scored[['session_id','user_id','risk_score','is_attack','attack_type']].head()


In [ ]:
y_true = test_scored['is_attack'].astype(int).to_numpy()
y_score = test_scored['risk_score'].astype(float).to_numpy()

roc_auc = roc_auc_score(y_true, y_score)
precision, recall, _ = precision_recall_curve(y_true, y_score)
pr_auc = auc(recall, precision)
ops = model.get_operating_points()

summary = pd.DataFrame([
    {'metric': 'ROC AUC', 'value': roc_auc},
    {'metric': 'PR AUC', 'value': pr_auc},
    {'metric': 'block_mode_threshold', 'value': ops.get('block_mode', {}).get('threshold', float('nan'))},
    {'metric': 'friction_mode_threshold', 'value': ops.get('friction_mode', {}).get('threshold', float('nan'))},
])
summary


In [ ]:
feature_importance = model.feature_importance()
figs = build_metrics_dashboard(
    test_scored,
    score_col='risk_score',
    target_col='is_attack',
    feature_importances=feature_importance,
)
figs['roc']


In [ ]:
figs['pr']


In [ ]:
figs['feature_importance']


In [ ]:
figs['cohort_analysis']


In [ ]:
figs['score_distribution']


In [ ]:
threshold_table = threshold_metrics_table(test_scored['is_attack'], test_scored['risk_score'])
best_recall_at_high_precision = (
    threshold_table[threshold_table['precision'] >= 0.95]
    .sort_values('recall', ascending=False)
    .head(5)
)
best_precision_at_high_recall = (
    threshold_table[threshold_table['recall'] >= 0.95]
    .sort_values('precision', ascending=False)
    .head(5)
)
best_recall_at_high_precision, best_precision_at_high_recall


In [ ]:
figs['threshold_slider']


## SQL Query Examples
```sql
-- Top risky sessions in the last 24h
SELECT session_id, user_id, risk_score, attack_type
FROM scored_sessions
WHERE event_ts >= NOW() - INTERVAL '24 hours'
ORDER BY risk_score DESC
LIMIT 100;

-- New-account fraud candidates
SELECT session_id, user_id, account_age_days, risk_score
FROM scored_sessions
WHERE account_age_days < 1
  AND risk_score >= 0.85
ORDER BY risk_score DESC;

-- Friction cohort for challenge flow
SELECT user_id, COUNT(*) AS sessions, AVG(risk_score) AS avg_risk
FROM scored_sessions
WHERE risk_score BETWEEN 0.40 AND 0.85
GROUP BY user_id
HAVING COUNT(*) >= 3
ORDER BY avg_risk DESC;
```